# 🎙️ Gerador de Áudio da Bíblia (Voz Clonada "Lucas" • OmniVoice)

### Como usar:
1. No menu superior do Google Colab, vá em **Ambiente de execução → Alterar o tipo de ambiente de execução** e confirme que está marcado **GPU T4**.
2. No seu **Google Drive**, crie uma pasta chamada **`BibliaAudio`** na raiz (`Meu Drive/BibliaAudio`) e coloque lá dentro o arquivo **`lucas_voz.aac`** (que está na pasta `colab/` do seu projeto).
3. Clique no botão **Play (▶)** nas células abaixo em ordem!

In [ ]:
# @title 1️⃣ Conectar Google Drive e instalar OmniVoice (rode 1 vez por sessão)
from google.colab import drive
drive.mount('/content/drive')

!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip install -q omnivoice soundfile
print('✅ Ambiente configurado com sucesso!')

In [ ]:
# @title 2️⃣ Carregar Modelo na GPU e preparar a voz "Lucas"
import os
import subprocess
import torch
from omnivoice import OmniVoice

DRIVE_DIR = '/content/drive/MyDrive/BibliaAudio'
os.makedirs(DRIVE_DIR, exist_ok=True)

RAW_VOICE_PATH = os.path.join(DRIVE_DIR, 'lucas_voz.aac')
REF_WAV_PATH = os.path.join(DRIVE_DIR, 'lucas_ref_18s.wav')

if not os.path.exists(RAW_VOICE_PATH):
    raise FileNotFoundError(
        f'Arquivo não encontrado: {RAW_VOICE_PATH}\n'
        '👉 Envie o arquivo colab/lucas_voz.aac para a pasta Meu Drive/BibliaAudio no seu Google Drive!'
    )

# Converte os primeiros 18s (sem silêncios longos) para WAV 24kHz mono (ideal para o OmniVoice)
if not os.path.exists(REF_WAV_PATH):
    subprocess.run([
        'ffmpeg', '-y', '-i', RAW_VOICE_PATH,
        '-af', 'silenceremove=start_periods=1:start_silence=0.2:start_threshold=-45dB',
        '-t', '18', '-ar', '24000', '-ac', '1', REF_WAV_PATH
    ], check=True)
    print('✅ Áudio de referência otimizado criado em:', REF_WAV_PATH)

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'🚀 Carregando k2-fsa/OmniVoice em {device}...')
model = OmniVoice.from_pretrained(
    'k2-fsa/OmniVoice',
    device_map=device,
    dtype=torch.float16 if device.startswith('cuda') else torch.float32
)

# Pré-calcula o embedding da voz 1 única vez para acelerar todos os capítulos!
print('🎙️ Extraindo assinatura de voz (voice_clone_prompt)...')
try:
    voice_prompt = model.create_voice_clone_prompt(ref_audio=REF_WAV_PATH)
    USE_CACHED_PROMPT = True
    print('✅ Voz Lucas pronta em cache (modo ultra-rápido ativado)!')
except Exception as e:
    USE_CACHED_PROMPT = False
    print('ℹ️ Usando ref_audio direto:', e)

In [ ]:
# @title 3️⃣ Gerar Capítulos e salvar direto no Google Drive (pula os que já existem)
import json
import re
import time
import unicodedata
import urllib.request
import numpy as np
import soundfile as sf

# ================= CONFIGURAÇÕES =================
VERSAO = 'naa'          # 'naa', 'nvi' ou 'acf'
FILTRO_LIVRO = ''       # Ex: 'Gênesis' ou 'Salmos' (deixe '' para gerar todos os livros em sequência)
FILTRO_CAPITULO = 0     # Ex: 23 (deixe 0 para gerar todos os capítulos)
LIMITE_CAPITULOS = 0    # Ex: 20 (deixe 0 para rodar sem limite até você parar)
MAX_CHARS_BLOCO = 800   # Na GPU T4 de 16GB do Colab, 800 caracteres roda rápido e seguro
# =================================================

def normalize(s):
    s = unicodedata.normalize('NFD', s.lower())
    return ''.join(c for c in s if unicodedata.category(c) != 'Mn').strip()

def slugify(s):
    return re.sub(r'^-+|-+$', '', re.sub(r'[^a-z0-9]+', '-', normalize(s)))

def split_chapter(header, verses, max_chars=MAX_CHARS_BLOCO):
    chunks = []
    current = header
    for v in verses:
        clean = v.strip()
        if not clean:
            continue
        if len(f'{current} {clean}') > max_chars:
            if current:
                chunks.append(current)
            if len(clean) > max_chars:
                clauses = re.split(r'(?<=[.;:,])\s+', clean)
                sub = ''
                for cl in clauses:
                    if len(f'{sub} {cl}'.strip()) > max_chars:
                        if sub:
                            chunks.append(sub)
                        sub = cl
                    else:
                        sub = f'{sub} {cl}'.strip() if sub else cl
                current = sub
            else:
                current = clean
        else:
            current = f'{current} {clean}' if current else clean
    if current:
        chunks.append(current)
    return chunks

# Baixa o JSON da Bíblia direto do seu GitHub
bible_url = f'https://raw.githubusercontent.com/lucasalbert-89/bible-reader/main/public/bibles/{VERSAO}.json'
print(f'📖 Carregando {VERSAO.upper()} de {bible_url}...')
with urllib.request.urlopen(bible_url) as resp:
    books = json.loads(resp.read().decode('utf-8'))

out_dir = os.path.join(DRIVE_DIR, VERSAO)
os.makedirs(out_dir, exist_ok=True)

total_chapters = sum(len(b['chapters']) for b in books)
existing_count = len([f for f in os.listdir(out_dir) if f.endswith('.mp3')])
print(f'📊 Progresso atual no Google Drive ({VERSAO.upper()}): {existing_count} / {total_chapters} capítulos prontos.\n')

target_book = normalize(FILTRO_LIVRO) if FILTRO_LIVRO else ''
generated_now = 0

for book in books:
    if target_book and normalize(book['name']) != target_book:
        continue
    slug = slugify(book['name'])

    for idx, verses in enumerate(book['chapters']):
        cap = idx + 1
        if FILTRO_CAPITULO and cap != FILTRO_CAPITULO:
            continue

        mp3_path = os.path.join(out_dir, f'{slug}-{cap}.mp3')
        if os.path.exists(mp3_path) and os.path.getsize(mp3_path) > 0:
            continue

        header = f"{book['name']}, capítulo {cap}."
        chunks = split_chapter(header, verses)
        t0 = time.time()
        print(f'⏳ Gerando {VERSAO.upper()} • {book["name"]} {cap} ({len(chunks)} blocos)...', end=' ', flush=True)

        wav_parts = []
        for chunk in chunks:
            if USE_CACHED_PROMPT:
                audio = model.generate(text=chunk, voice_clone_prompt=voice_prompt)
            else:
                audio = model.generate(text=chunk, ref_audio=REF_WAV_PATH)
            item = audio[0]
            wav_np = item.detach().cpu().numpy().squeeze() if hasattr(item, 'detach') else np.asarray(item).squeeze()
            wav_parts.append(wav_np)
            # Pequeno silêncio de 0.35s entre blocos para soar natural
            wav_parts.append(np.zeros(int(24000 * 0.35), dtype=np.float32))
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        full_wav = np.concatenate(wav_parts)
        tmp_wav = '/content/temp_chapter.wav'
        sf.write(tmp_wav, full_wav, 24000)

        # Converte para MP3 otimizado para voz (64 kbps mono) e salva direto no Google Drive
        subprocess.run([
            'ffmpeg', '-y', '-i', tmp_wav,
            '-codec:a', 'libmp3lame', '-b:a', '64k', '-ac', '1', mp3_path
        ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        dt = time.time() - t0
        kb = os.path.getsize(mp3_path) // 1024
        generated_now += 1
        existing_count += 1
        pct = (existing_count / total_chapters) * 100
        print(f'✅ OK ({kb} KB em {dt:.1f}s) | Total: {existing_count}/{total_chapters} ({pct:.1f}%)')

        if LIMITE_CAPITULOS and generated_now >= LIMITE_CAPITULOS:
            print(f'\n🎯 Limite de {LIMITE_CAPITULOS} capítulos atingido nesta execução!')
            break
    if LIMITE_CAPITULOS and generated_now >= LIMITE_CAPITULOS:
        break

print(f'\n🎉 Sessão finalizada! Capítulos gerados agora: {generated_now}. Total no Drive: {existing_count}/{total_chapters}.')

In [ ]:
# @title 4️⃣ Enviar todos os MP3s do Google Drive direto para o Hugging Face
from huggingface_hub import HfApi

HF_REPO_ID = 'lucasalbert89/bible-audio'
HF_TOKEN = 'hf_COLE_SEU_TOKEN_WRITE_AQUI'

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=HF_REPO_ID, repo_type='dataset', exist_ok=True)

print(f'☁️ Enviando arquivos de {DRIVE_DIR} para https://huggingface.co/datasets/{HF_REPO_ID} ...')
api.upload_folder(
    folder_path=DRIVE_DIR,
    repo_id=HF_REPO_ID,
    repo_type='dataset',
    allow_patterns=['naa/*.mp3', 'nvi/*.mp3', 'acf/*.mp3'],
    commit_message='Atualizar capítulos de áudio da Bíblia'
)
print('✅ Upload concluído! Todos os capítulos prontos já estão no ar!')